# Luna component study: 100 offline trajectories

Compare **0, 2 and 4 demonstrations** on the same held-out 12×12×4 trajectories. This notebook only reads results and draws plots; it does not call Luna.

- **CLLP:** exact offline target reached within four primitive actions.
- **Generator + CLLP:** valid proposed board and whether the same controller reaches it. Failure to reach is not proof of mathematical unreachability.
- **Value:** non-decreasing and strictly increasing adjacent predictions, ties, and value along each witness trajectory.

Errors and missing cases do not count as successes or failures; hover over bars to see their denominators. Confidence intervals use Wilson intervals for binary rates and a trajectory bootstrap for per-trajectory mean rates.

In [ ]:
import sys
from pathlib import Path

anchors = (Path.cwd(), *Path.cwd().parents, *Path(sys.executable).parents)
ROOT = next((candidate.resolve() for anchor in anchors for candidate in (anchor, anchor / "CaRL")
             if (candidate / "carl/icl").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Cannot find CaRL; start the kernel in the workspace or checkout.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Imports follow checkout discovery so this notebook also works from the workspace root.
# pylint: disable=wrong-import-position
import pandas as pd  # noqa: E402 -- checkout path is configured above.
from IPython.display import display  # noqa: E402 -- checkout path is configured above.

from carl.icl.evaluation.component_study import export, figures, load_cases, summarize  # noqa: E402 -- checkout path is configured above.
from scripts.evaluate_component_study import load_study, plan  # noqa: E402 -- checkout path is configured above.

# pylint: enable=wrong-import-position

STUDY_CONFIG = ROOT / "configs/icl/component_study/luna_100.json"
study = load_study(STUDY_CONFIG)
result_dir = Path(study.output_dir)
if not result_dir.is_absolute():
    result_dir = ROOT / result_dir
print(f"Model: {study.model} | results: {result_dir}")
display(plan(study))

## Run the study

Run these commands in a terminal from `CaRL/`. The first is free. The second sends paid requests for cache misses; run it only when ready. Each case is saved independently, so interrupting and resuming preserves completed cases.

```bash
uv run --extra icl python scripts/evaluate_component_study.py --plan
uv run --extra icl python scripts/evaluate_component_study.py --execute
# After interruption:
uv run --extra icl python scripts/evaluate_component_study.py --execute --resume
# To retry recorded failures:
uv run --extra icl python scripts/evaluate_component_study.py --execute --resume --retry-errors
```

To run only a subset, use `--component policy`, `--component generator`, or `--component value`, and `--shots 0 2 4`. Use `--resume` to append subsequent subsets to the same study. Offline `--mock` is for pipeline checks only; keep its output directory separate from the live study.

In [ ]:
manifest_path = result_dir / "manifest.json"
if manifest_path.exists():
    manifest, cases = load_cases(result_dir)
    print(f"Provider: {manifest['provider']} | saved trajectory cases: {len(cases)}")
    display(pd.DataFrame(summarize(manifest, cases)))
else:
    manifest, cases = None, []
    print("No saved study yet. Run the commands above, then re-run this cell.")

## CLLP and generated subgoals

Generator validity checks the board representation and invariants. The conditional rate uses only valid boards with a completed controller rollout; the overall rate also includes cases with no valid board. A lower-bound measurement of reachability is controller-dependent.

In [ ]:
if manifest is not None:
    plots = figures(manifest, cases)
    plots["policy_reach"].show()
    plots["generator_reach"].show()

## Value monotonicity

Values are discounted from the estimated remaining-action count. Weak monotonicity counts ties as non-decreasing; strict increases and ties are shown separately so a constant predictor cannot look informative. The offline witness need not be a shortest path.

In [ ]:
if manifest is not None:
    plots = figures(manifest, cases)
    plots["value_monotonicity"].show()
    plots["value_trajectory"].show()

## Export

Exports standalone interactive HTML plots, `summary.csv`, `summary.json`, `usage.csv`, and a shared Plotly JavaScript file. Physical token and request counts are separate from logical requests and local cache hits.

In [ ]:
if manifest is not None:
    print(f"Exported: {export(result_dir)}")